# Dataset EDA & Results Analysis

Quick look at the bin segmentation dataset and training results.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from collections import Counter
from PIL import Image

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# load annotations
train_coco = COCO('../data/train/annotations.json')
val_coco = COCO('../data/val/annotations.json')
test_coco = COCO('../data/test/annotations.json')

print(f'Train: {len(train_coco.getImgIds())} images, {len(train_coco.getAnnIds())} annotations')
print(f'Val:   {len(val_coco.getImgIds())} images, {len(val_coco.getAnnIds())} annotations')
print(f'Test:  {len(test_coco.getImgIds())} images, {len(test_coco.getAnnIds())} annotations')

## Objects per image distribution

In [ ]:
# how many objects per image?
objs_per_img = []
for img_id in train_coco.getImgIds():
    ann_ids = train_coco.getAnnIds(imgIds=img_id, iscrowd=False)
    objs_per_img.append(len(ann_ids))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(objs_per_img, bins=range(0, max(objs_per_img) + 2), edgecolor='black', alpha=0.7)
ax.set_xlabel('Objects per image')
ax.set_ylabel('Count')
ax.set_title(f'Objects per image (train set, mean={np.mean(objs_per_img):.1f})')
ax.axvline(np.mean(objs_per_img), color='red', linestyle='--', label=f'mean={np.mean(objs_per_img):.1f}')
ax.legend()
plt.tight_layout()
plt.show()

## Object size distribution

In [ ]:
# object areas
areas = []
for ann_id in train_coco.getAnnIds():
    ann = train_coco.loadAnns(ann_id)[0]
    areas.append(ann['area'])

areas = np.array(areas)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(areas, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Mask area (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Object area distribution')

# log scale to see the tail
axes[1].hist(np.log10(areas + 1), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('log10(area)')
axes[1].set_ylabel('Count')
axes[1].set_title('Object area (log scale)')

plt.tight_layout()
plt.show()

print(f'Area stats: min={areas.min():.0f}, median={np.median(areas):.0f}, '
      f'mean={areas.mean():.0f}, max={areas.max():.0f}')

## Sample images

In [ ]:
# show a few sample images with masks overlaid
from pycocotools import mask as mask_utils

sample_ids = np.random.choice(train_coco.getImgIds(), size=4, replace=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, img_id in zip(axes.flat, sample_ids):
    img_info = train_coco.loadImgs(img_id)[0]
    img = Image.open(os.path.join('../data/train/images', img_info['file_name']))
    ax.imshow(img)
    
    ann_ids = train_coco.getAnnIds(imgIds=img_id)
    anns = train_coco.loadAnns(ann_ids)
    
    # overlay masks
    for ann in anns:
        if isinstance(ann['segmentation'], list):
            rle = mask_utils.frPyObjects(ann['segmentation'], img_info['height'], img_info['width'])
            rle = mask_utils.merge(rle)
        else:
            rle = ann['segmentation']
        m = mask_utils.decode(rle)
        color = np.random.random(3)
        overlay = np.zeros((*m.shape, 4))
        overlay[m > 0] = [*color, 0.4]
        ax.imshow(overlay)
    
    ax.set_title(f'{len(anns)} objects')
    ax.axis('off')

plt.suptitle('Sample training images with instance masks', fontsize=14)
plt.tight_layout()
plt.show()

## Ablation results comparison

Load from MLflow or manually enter if you recorded them.

In [ ]:
# manually entered from our runs — update these with your actual numbers
results = {
    'R50 baseline': {'mAP': 0.441, 'mask_iou': 0.689, 'pick_success': 0.712},
    'R101 baseline': {'mAP': 0.483, 'mask_iou': 0.721, 'pick_success': 0.748},
    'R101 no-occ': {'mAP': 0.412, 'mask_iou': 0.663, 'pick_success': 0.671},
    'R50 focal': {'mAP': 0.458, 'mask_iou': 0.704, 'pick_success': 0.730},
}

names = list(results.keys())
metrics = ['mAP', 'mask_iou', 'pick_success']
x = np.arange(len(names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, metric in enumerate(metrics):
    vals = [results[n][metric] for n in names]
    bars = ax.bar(x + i * width, vals, width, label=metric)
    # add value labels
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(names, rotation=15)
ax.set_ylabel('Score')
ax.set_title('Ablation Study Results')
ax.legend()
ax.set_ylim(0, 0.85)
plt.tight_layout()
plt.show()

### Key takeaways

- Occlusion augmentation is the biggest single lever — ~7% swing in pick success
- R101 vs R50 backbone: modest improvement (~4% mAP) for 2x the training time
- Focal loss helps on small/occluded objects but the overall gain is marginal over CE
- The pick-success proxy tracks mAP pretty linearly, which is encouraging